# SQRT (Sub-zero Quantitative Research Team)

## Post-Mission Data Analysis

**Author:** Caimin Keavney

The processes applied below will extract, translate and store all relevant mission data acquired by the SQRT mission. This data will be in the form of both transmitted data and data stored onboard the SD card. Depending on which of these modes is relevant, (depending on payload recovery) two workflows will be required to unload the data products.

### Onboard Storage (SD card)

In [47]:
import os
import numpy as np
import matplotlib.pyplot as plt

In [206]:
# The folder container all of the SD card is defined. 

def save_frames(times, data, out_dir):
    """
    This function produces and saves figures 
    """
    os.makedirs(out_dir, exist_ok=True)

    parsed_frames = []

    # The temperatures are read in from the file
    for line in data:
        values = [float(x) for x in line.strip().split(",") if x]
        parsed_frames.append(values)

    combined_frames = []

    # Each full frame is appended to the frame list
    for full_frame in parsed_frames:        
        if len(full_frame) in (768, 48):
            combined_frames.append(full_frame)

            

    # Plot and save frames
    for i, frame in enumerate(combined_frames):
        frame = np.array(frame)

        if len(frame) == 768:
            shaped = frame.reshape(24, 32)

        elif len(frame) == 48:
            shaped = frame.reshape(6,8)
        else:
            print(f"Bad combined frame size: {len(frame)}")
            continue

        fig, ax = plt.subplots()
        heatmap = ax.imshow(shaped, cmap="coolwarm", origin="upper")
        plt.colorbar(heatmap)

        label = times[i] if i < len(times) else f"frame_{i}"
        ax.set_title(label)

        filename = os.path.join(out_dir, f"frame_{label}.png")
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)

def science_data_proc(datafile):
    """
    Extracts and cleans Science Data from MLX90640.
    """
    times = []
    frame_arrays = []
    with open(datafile, "r") as f:
        # The data is unloaded from the file
        data = f.readlines()[1:]

    # The times and data are stored in two separate lists.
    for line in data:
        if len(line) < 1000:
            time = line.split()[0]
            times.append(time)
        else:
            frame_arrays.append(line)

    return times, frame_arrays


def house_proc(housefile):
    """
    Extracts and stores Housekeeping Data from sensors
    """
    Housekeeping_Data = {
        "Times": [],
        "Internal_Temperature": [],
        "External_Temperature": [],
        "Pressure": [],
        "Latitudes": [],
        "Longitudes": [],
        "Altitudes": [],
        "HDOP": []
    }

    with open(housefile, "r") as f:
        lines = f.readlines()[1:]

    
    for line in lines:
        items = line.strip().split(",")
        
        if len(items) == 9:
            Housekeeping_Data["Times"].append(items[0])
            Housekeeping_Data["Pressure"].append(items[2])
            Housekeeping_Data["External_Temperature"].append(items[3])
            Housekeeping_Data["Internal_Temperature"].append(items[4])
            Housekeeping_Data["Latitudes"].append(items[5])
            Housekeeping_Data["Longitudes"].append(items[6])
            Housekeeping_Data["Altitudes"].append(items[7])
            Housekeeping_Data["HDOP"].append(items[8])
        else:
            continue

    return Housekeeping_Data
    

    
def error_proc(error_file):
    """
    Extracts and stores Error Log
    """
    error_dict = {
        "Times": [],
        "Error": [],
        "Sensor/Class": []
    }

    with open(error_file, "r") as f:
        lines = f.readlines()[1:]

    for line in lines:
        items = line.strip().split(",")
        if len(items) == 3:     
            error_dict["Times"].append(items[0])
            error_dict["Error"].append(items[1])
            error_dict["Sensor/Class"].append(items[2])
        else:
            continue
    
    return error_dict

def trig_proc(trig_file):
    """
    Extracts and stores Trigger file information (trigger statuses & trigger conditions if applicable).
    """

    trig_dic = {
        "time": [],
        "trig_statuses": [],
        "trig_conditions": [],
        "trig_pressures": [],
        "trig_altitudes": []
    }

    with open(trig_file, "r") as f:
        lines = f.readlines()[1:]

    for line in lines:
        items = line.strip().split(",")
        if len(items) ==5:
            trig_dic["time"].append(items[0])
            trig_dic["trig_statuses"].append(items[1])
            trig_dic["trig_conditions"].append(items[2])
            trig_dic["trig_pressures"].append(items[3])
            trig_dic["trig_altitudes"].append(items[4])
        else:
            continue
        
    return trig_dic


def sd_processor(folder_path):
    files = os.listdir(folder_path)
    
    for file in files:
        fullpath = folder_path + "/" + file
        
        if "data" in file:
            t, d = science_data_proc(fullpath)
            
            if "trigger" in file:
                save_frames(t, d, "Post_Trigger_Frames")
            else:
                save_frames(t, d, "Pre_Trigger_Frames")

        elif "housekeeping" in file:
            house_dict = house_proc(fullpath)

        elif "error" in file:
            error_dict = error_proc(fullpath)

        elif "trigger_log" in file: 
            trig_dict = trig_proc(fullpath)

        else: 
            continue
    return house_dict, error_dict, trig_dict

### Transmitted Data Packets Analysis

In [210]:
all_house_data = []
all_science_data = []


def packet_proc(textfile):
    """
    Deals with all packets 
    """
    with open(textfile, "r") as f:
        packets = f.readlines()

    for packet in packets:
        packet= packet.strip()
        
        items = packet.split("|")

        
        if items[1].strip() == "SQRT":
            packet_type = items[0].strip()[-1]
            
            if packet_type == "T":
                all_house_data.append(packet)    # Telemetry Packet
                
            elif packet_type == "D":
                all_science_data.append(packet)   # Science Packet

        else:
            continue
    house_info = house_packet_parser(all_house_data)
    frames, science_info = science_packet_parser(all_science_data)

    decoded_frames = frames_proc(frames)
    
    return house_info, science_info, decoded_frames

def house_packet_parser(packets):
    """
    Parses data from received data packets and stores/ housekeeping/science data
    """
    house_data = {
        "timestamp": [],
        "pressure": [],
        "external_temp": [],
        "internal_temp": [],
        "latitude": [],
        "longitude": [],
        "altitude": [],
        "hdop": [],
        "error_count": [],
        "error_types": []
    }
    
    for packet in packets:
        
        items = packet.split("|")
        
        if len(items)<14:
            continue
            
        house_data["timestamp"].append(items[4])
        house_data["latitude"].append(items[5])
        house_data["longitude"].append(items[6])
        house_data["hdop"].append(items[7])
        house_data["altitude"].append(items[8])
        house_data["internal_temp"].append(items[9])
        house_data["external_temp"].append(items[10])
        house_data["pressure"].append(items[11])
        house_data["error_count"].append(items[12])
        house_data["error_types"].append(items[13])


    return house_data

def science_packet_parser(packets):
    """
    Handles Science data packets including frame data, which is sent to save_data and saved 
    """
    frames_data = []

    trig_dic = {
        "trig": [],
        "condition": [],
        "pres": [],
        "alt": []
    }

    for packet in packets:
        items = packet.split("|")

        frames_data.append(items[2])
        trig_dic["trig"].append(items[3])
        trig_dic["condition"].append(items[4])
        trig_dic["pres"].append(items[5])
        trig_dic["alt"].append(items[6])

    return frames_data, trig_dic
    
def frames_proc(frames_list):
    """
    Processes frame data, which is in binary format
    """
    frames = []
    for frame in frames_list:
        arr = array('h')
        arr.frombytes(bytes.fromhex(frame))
        decoded_frame = [x/100 for x in arr]
        
        frames.append(decoded_frame)

        
    return frames